# Video Process Pipeline

End-to-end pipeline that takes raw videos and produces a training-ready dataset.

**Steps:**
1. **Analyze** raw videos in `data/raw/` (resolution, fps, duration, brightness)
2. **Process** videos (resize to 768x512, crop, re-encode at 24 fps, trim to 5 s max)
3. **Preview** thumbnails for visual QA
4. **Auto-caption** with Gemini 2.5 Flash
5. **Validate** the final dataset

**Outputs** (in `data/processed/`):
- `videos/` -- processed `.mp4` files
- `captions.json` -- filename-to-caption mapping
- `videos.txt` -- one video path per line
- `prompts.txt` -- one caption per line (same order)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.config import (
    CAPTION_CONTEXT,
    DATASET_DIR,
    GEMINI_API_KEY,
    MAX_DURATION_SEC,
    MIN_DURATION_SEC,
    MIN_FRAMES,
    PER_VIDEO_CONTEXT,
    RAW_VIDEO_DIR,
    TARGET_FPS,
    TARGET_HEIGHT,
    TARGET_WIDTH,
    TRIGGER_TOKEN,
    VIDEO_DIR,
)
from dotenv import load_dotenv
load_dotenv()
from src.data.analyze import analyze_all
from src.data.process import process_all
from src.data.preview import preview_videos
from src.captioning.gemini import caption_all
from src.training.validate import validate_dataset

Path(str(DATASET_DIR)).mkdir(parents=True, exist_ok=True)
Path(str(VIDEO_DIR)).mkdir(parents=True, exist_ok=True)

print("Configuration")
print("=" * 40)
print(f"Raw video dir : {RAW_VIDEO_DIR}")
print(f"Dataset dir   : {DATASET_DIR}")
print(f"Video dir     : {VIDEO_DIR}")
print(f"Target        : {TARGET_WIDTH}x{TARGET_HEIGHT} @ {TARGET_FPS} fps")
print(f"Duration      : {MIN_DURATION_SEC}–{MAX_DURATION_SEC} s")
print(f"Min frames    : {MIN_FRAMES}")
print(f"Trigger token : {TRIGGER_TOKEN}")

In [ ]:
import pandas as pd

print("STEP 1: Analyzing raw videos")
print("=" * 60)

analysis = analyze_all(str(RAW_VIDEO_DIR), min_duration=MIN_DURATION_SEC)

if not analysis:
    raise RuntimeError(f"No valid videos found in {RAW_VIDEO_DIR}")

df_analysis = pd.DataFrame(analysis)
df_analysis["resolution"] = df_analysis["width"].astype(str) + "x" + df_analysis["height"].astype(str)

display_cols = ["filename", "resolution", "fps", "duration_sec", "mean_brightness", "is_likely_black"]
pd.set_option("display.max_colwidth", None)
display(df_analysis[display_cols])

In [ ]:
print("STEP 2: Processing videos (resize, crop, trim)")
print("=" * 60)

processed = process_all(
    video_analysis=analysis,
    output_dir=str(VIDEO_DIR),
    target_w=TARGET_WIDTH,
    target_h=TARGET_HEIGHT,
    target_fps=TARGET_FPS,
    max_duration=MAX_DURATION_SEC,
    min_duration=MIN_DURATION_SEC,
    min_frames=MIN_FRAMES,
)

if not processed:
    raise RuntimeError("No videos survived processing.")

df_processed = pd.DataFrame([
    {
        "original": v["original"],
        "processed": v["processed"],
        "resolution": f"{v['info']['width']}x{v['info']['height']}",
        "fps": v["info"]["fps"],
        "frames": v["info"]["frame_count"],
        "duration_sec": v["info"]["duration_sec"],
    }
    for v in processed
])
display(df_processed)

In [ ]:
%matplotlib inline

print("STEP 3: Preview thumbnails")
print("=" * 60)

preview_videos(processed, TARGET_WIDTH, TARGET_HEIGHT, TARGET_FPS)

In [ ]:
import json
import os

print("STEP 4: Auto-captioning with Gemini")
print("=" * 60)

if not GEMINI_API_KEY:
    raise RuntimeError(
        "GEMINI_API_KEY is not set. Add it to your .env file at the project root."
    )

captions = caption_all(
    video_dir=str(VIDEO_DIR),
    dataset_dir=str(DATASET_DIR),
    trigger_token=TRIGGER_TOKEN,
    context=CAPTION_CONTEXT,
    api_key=GEMINI_API_KEY,
    per_video_context=PER_VIDEO_CONTEXT,
)

df_captions = pd.DataFrame(
    [(fn, cap) for fn, cap in captions.items()],
    columns=["filename", "caption"],
)
df_captions["status"] = df_captions["caption"].apply(
    lambda c: "FAILED" if c.startswith("[FAILED]") else "OK"
)

total = len(df_captions)
success = (df_captions["status"] == "OK").sum()
failed = total - success

print(f"\nTotal: {total}  |  Success: {success}  |  Failed: {failed}")

if failed > 0:
    print("\nFAILED captions:")
    display(df_captions[df_captions["status"] == "FAILED"])

display(df_captions[["filename", "caption"]])

In [ ]:
print("STEP 5: Validating dataset")
print("=" * 60)

is_valid = validate_dataset(
    dataset_dir=str(DATASET_DIR),
    target_w=TARGET_WIDTH,
    target_h=TARGET_HEIGHT,
    min_frames=MIN_FRAMES,
)

if not is_valid:
    raise RuntimeError("Dataset has errors. Fix them before training.")

videos_txt = os.path.join(str(DATASET_DIR), "videos.txt")
prompts_txt = os.path.join(str(DATASET_DIR), "prompts.txt")

with open(videos_txt) as f:
    n_videos = len(f.read().strip().splitlines())
with open(prompts_txt) as f:
    n_prompts = len(f.read().strip().splitlines())

assert n_videos == n_prompts, (
    f"Line count mismatch: videos.txt has {n_videos}, prompts.txt has {n_prompts}"
)

print(f"\nvideos.txt:  {n_videos} entries")
print(f"prompts.txt: {n_prompts} entries")
print(f"\nOutput directory: {DATASET_DIR}")
print("\nDataset is ready for training.")